In [0]:
%python
# dml/02b_carga_stg_cvm_informe_ativo_passivo.ipynb
# MAGIC %pip install requests pandas numpy # Garante as dependencias no compute Serverless

# %%
import requests
import zipfile
import io
import pandas as pd
import numpy as np
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# %%
# 1. Recupera os de-paras de CNPJ e Ticker na tabela dim_fundo_imobiliario
catalogo = "product_dev"
schema = "financas"
tabela_dim = "dim_fundo_imobiliario"

print(f"Buscando CNPJs ativos mapeados na tabela {catalogo}.{schema}.{tabela_dim}...")

# Força o fuso horário de Brasília para garantir consistência de datas
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

# Lê apenas fundos que possuem CNPJ mapeados na dimensão
df_dim_ativos = spark.sql(f"""
    SELECT ticker, cnpj 
    FROM {catalogo}.{schema}.{tabela_dim} 
    WHERE cnpj IS NOT NULL
""")

print(f"Total de FIIs ativos com CNPJ para processamento: {df_dim_ativos.count()}")

# %%
# 2. Faz o download do ZIP de 2026 diretamente da CVM
url_zip = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/inf_mensal_fii_2026.zip"
headers = {'User-Agent': 'Mozilla/5.0'}

print(f"Baixando pacote anual de Informes Mensais da CVM (isto pode levar de 15s a 30s)...")
response = requests.get(url_zip, headers=headers, timeout=30)
response.raise_for_status()
print("Download do ZIP concluído com sucesso!")

# %%
# 3. Abre o ZIP e extrai o arquivo de Ativo e Passivo da CVM (Tratamento Defensivo)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))
arquivo_ativo_passivo = "inf_mensal_fii_ativo_passivo_2026.csv"

print(f"Lendo e parseando o arquivo de balanço patrimonial {arquivo_ativo_passivo} em Pandas...")

with zip_file.open(arquivo_ativo_passivo) as f:
    # Lógica defensiva para ler o arquivo mesmo com eventuais linhas desformatadas
    df_pd_ativo_passivo = pd.read_csv(
        f, 
        sep=';', 
        encoding='ISO-8859-1', 
        on_bad_lines='skip'
    )

print(f"Total de registros de ativos/passivos brutos lidos: {len(df_pd_ativo_passivo)}")

# %%
# 4. Transforma para Spark DataFrame e executa a Regra Contabil (Último Mês/Versão)
print("Processando dados de composição de ativos no Spark...")

# Converte de Pandas para Spark de forma nativa e força strings temporárias para limpeza
df_spark_raw = spark.createDataFrame(df_pd_ativo_passivo.astype(str))

# Janela contábil particionada por CNPJ e ordenada pelo mês/versão mais recentes
janela_competencia = Window.partitionBy("CNPJ_Fundo_Classe").orderBy(
    F.col("Data_Referencia").desc(), 
    F.col("Versao").desc()
)

df_ativo_passivo_filtrado = (
    df_spark_raw
    # Limpa as formatações do CNPJ CVM para cruzamento
    .withColumn("cnpj_limpo", F.regexp_replace(F.col("CNPJ_Fundo_Classe"), r"[\./-]", ""))
    
    # Aplica o Row Number na janela para pegar apenas a competência e versão mais recentes de cada fundo
    .withColumn("rn", F.row_number().over(janela_competencia))
    .filter(F.col("rn") == 1)
    
    # Tratamento defensivo de conversões numéricas para evitar erros de casting nulo (NaN -> NULL)
    .withColumn("tot_investido", F.coalesce(F.col("Total_Investido").cast("double"), F.lit(0.0)))
    .withColumn("imoveis_renda", F.coalesce(F.col("Imoveis_Renda_Acabados").cast("double"), F.lit(0.0)))
    .withColumn("imoveis_desenv", F.coalesce((F.col("Terrenos").cast("double") + F.col("Imoveis_Renda_Construcao").cast("double")), F.lit(0.0)))
    .withColumn("caixa_disp", F.coalesce(F.col("Disponibilidades").cast("double"), F.lit(0.0)))
    .withColumn("invest_cri_cra", F.coalesce((F.col("CRI").cast("double") + F.col("CRI_CRA").cast("double")), F.lit(0.0)))
    .withColumn("invest_lci_lca", F.coalesce((F.col("Letras_Hipotecarias").cast("double") + F.col("LCI").cast("double") + F.col("LCI_LCA").cast("double")), F.lit(0.0)))
    .withColumn("invest_outros_fiis", F.coalesce(F.col("FII").cast("double"), F.lit(0.0)))
    .withColumn("invest_acoes", F.coalesce(F.col("Acoes").cast("double"), F.lit(0.0)))
    .withColumn("tot_passivo", F.coalesce(F.col("Total_Passivo").cast("double"), F.lit(0.0)))
)

# %%
# 5. Junta os dados de balanço CVM com os Tickers ativos da nossa Dimensão de FIIs
print("Associando dados de ativos da CVM com os Tickers ativos da carteira...")

df_ativo_passivo_final = (
    df_ativo_passivo_filtrado.alias("cvm")
    .join(
        df_dim_ativos.alias("dim"),
        F.col("cvm.cnpj_limpo") == F.col("dim.cnpj"),
        "inner" # INNER JOIN garante que só vamos salvar o que temos na dimensão!
    )
    .select(
        F.col("dim.ticker").alias("ticker"),
        F.col("cvm.Data_Referencia").cast("date").alias("data_referencia"),
        F.col("cvm.cnpj_limpo").alias("cnpj"),
        F.col("cvm.tot_investido").alias("total_investido"),
        F.col("cvm.imoveis_renda").alias("valor_imoveis_renda"),
        F.col("cvm.imoveis_desenv").alias("valor_imoveis_desenvolvimento"),
        F.col("cvm.caixa_disp").alias("valor_caixa_disponibilidades"),
        F.col("cvm.invest_cri_cra").alias("valor_investido_cri_cra"),
        F.col("cvm.invest_lci_lca").alias("valor_investido_lci_lca"),
        F.col("cvm.invest_outros_fiis").alias("valor_investido_outros_fiis"),
        F.col("cvm.invest_acoes").alias("valor_investido_acoes"),
        F.col("cvm.tot_passivo").alias("total_passivo_obrigacoes"),
        F.from_utc_timestamp(F.current_timestamp(), "America/Sao_Paulo").alias("data_carga")
    )
)

# Cria a view temporária no Spark
df_ativo_passivo_final.createOrReplaceTempView("temp_cvm_ativo_passivo_consolidado")

# %%
# 6. Execução do INSERT OVERWRITE dinâmico na tabela de Staging de Ativo/Passivo CVM
tabela_destino = "stg_cvm_informe_ativo_passivo"

qry_insert_stg = f"""
  INSERT OVERWRITE {catalogo}.{schema}.{tabela_destino}
  SELECT 
    ticker,
    data_referencia,
    cnpj,
    total_investido,
    valor_imoveis_renda,
    valor_imoveis_desenvolvimento,
    valor_caixa_disponibilidades,
    valor_investido_cri_cra,
    valor_investido_lci_lca,
    valor_investido_outros_fiis,
    valor_investido_acoes,
    total_passivo_obrigacoes,
    data_carga
  FROM temp_cvm_ativo_passivo_consolidado
"""

print(f"Gravando dados consolidados de balanço patrimonial em: {catalogo}.{schema}.{tabela_destino}...")

# Executa a gravação atômica
spark.sql(qry_insert_stg)

print("✅ Carga das tabelas de balanço Silver de Ativos/Passivos CVM concluída com SUCESSO!")